In [1]:
# ML Model for IP Analyses - Part 5
# Creating Model with Logistic Regression 

# Importing Libraries for ML
import pandas as pd
from sklearn import linear_model
import numpy as np

import os
import tldextract as tld

In [2]:
### Switching Path
data_path = "Users/Nyk/NCI/OneDrive - National College of Ireland/NCI/Year 4/3 Final Year Project/S3/0. Deliverables/Final Submission (AUG 3)/2. Code/ML Building/Jupyter"
os.chdir(os.getcwd() + '/imported_data/structured_data/')


In [3]:
os.getcwd()

'D:\\Users\\Nyk\\NCI\\OneDrive - National College of Ireland\\NCI\\Year 4\\3 Final Year Project\\S3\\0. Deliverables\\Final Submission (AUG 3)\\2. Code\\ML Building\\Jupyter\\imported_data\\structured_data'

In [4]:
# View the Contents of Folder
os.listdir()

['structured_data.csv']

In [5]:
# Importing Data
data = pd.read_csv('structured_data.csv')

In [6]:
# View Data Contents
data.head()

,Unnamed: 0,URL,Threat,Length,Contains_HTTPS,Has_IP,Digit_Count,Dot_Count,Has_Subdomain,Subdomain_Count,Hyphen_Count,Special_Count
0,0,https://www.repvtrhsrbpku.ug,1,28,1,0,0,2,0,0,0,1
1,1,https://www.cbbridal.com,0,24,1,0,0,2,0,0,0,1
2,2,https://www.zrtvxqyqtslf.info,1,29,1,0,0,2,0,0,0,1
3,3,https://www.hrkyraongvqnaxfct.pw,1,32,1,0,0,2,0,0,0,1
4,4,https://www.volcengine-cache-weboffice.wpscdn.cn,0,48,1,0,0,3,1,1,2,3


In [7]:
### Adding Typosquating

In [8]:
# Libraries used:
# 1. tldextract - for obtaining domain names
# 2. Levenshtein - for similarity comparison

In [9]:
# Importing a list of popular domains:
domains_list_path_win = "D:/Users/Nyk/NCI/OneDrive - National College of Ireland/NCI/Year 4/3 Final Year Project/S3/0. Deliverables/Final Submission (AUG 3)/2. Code/ML Building/Jupyter/imported_data"
os.chdir(domains_list_path_win)
popular_domains = pd.read_csv('top-1m.csv')

In [10]:
popular_domains.head()

,1,google.com
0,2,youtube.com
1,3,facebook.com
2,4,baidu.com
3,5,wikipedia.org
4,6,yahoo.com


In [11]:
popular_domains.drop(columns={'1'})

,google.com
0,youtube.com
1,facebook.com
2,baidu.com
3,wikipedia.org
4,yahoo.com
...,...
999994,sibf.org
999995,bukapintu.co
999996,klatovynet.cz
999997,elconquistadorfm.cl


In [12]:
popular_domains.columns = ['ID','Domain']
popular_domains.head()

,ID,Domain
0,2,youtube.com
1,3,facebook.com
2,4,baidu.com
3,5,wikipedia.org
4,6,yahoo.com


In [13]:
popular_domains.drop(columns={'ID'})

,Domain
0,youtube.com
1,facebook.com
2,baidu.com
3,wikipedia.org
4,yahoo.com
...,...
999994,sibf.org
999995,bukapintu.co
999996,klatovynet.cz
999997,elconquistadorfm.cl


In [14]:
os.listdir()

['bad_url.csv',
 'cleaned_csv',
 'dga.csv',
 'good_csv',
 'phish.csv',
 'phish_clean.csv',
 'structured_data',
 'top-1m.csv',
 'top_cleaned.csv',
 'typosquatting_list',
 'url.csv']

In [15]:
# Saving Changes:
popular_domains.to_csv('top_cleaned.csv')

In [16]:
data['Host_in_Subdomain'] = 0
data['Host_in_Domain'] = 0
data['Similarity'] = 0

In [17]:
import Levenshtein

In [18]:

def extract_data(url, name):
    extracted = tld.extract(url)
    if name == 'hostname':
        return extracted.domain.lower()
    elif name == 'subdomain':
        return extracted.subdomain.lower()
    else:
        print('Type not found')
        return None

def check_domain(url):
     try:
         hostname  = extract_data(url,'hostname')
         if hostname in popular_domains['Domain'].values:
             return 1
         else:
             return 0
     except Exception as e:
         print('Exception occured in check_domain method')
         print(f'Exception: {e}')

def check_subdomain(url):
    try:
        hostname = extract_data(url,'hostname')
        subdomain = extract_data(url,'subdomain')
        for domain in popular_domains['Domain']:
            domain_name = domain.split('.')[0]
            if domain_name in subdomain and hostname != domain_name:
                return 1
            else:
                return 0
    except Exception as e:
         print('Exception occured in check_subdomain method')
         print(f'Exception: {e}')
            
def check_similarity(url):
    try:
        hostname = extract_data(url,'hostname')

        best_score = 0
        for domain in popular_domains['Domain']:
            similarity_score = Levenshtein.ratio(domain, hostname)
            if similarity_score > best_score:
                return similarity_score
            else:
                return best_score
    except Exception as e:
        print('Exception occured in check_similarity method')
        print(f'Exception: {e}')

In [ ]:
for index in data.index:
    result = check_domain(data.loc[index, 'URL'])
    data.loc[index, 'Host_in_Domain'] = result

In [ ]:
data.head()